#Embedding-Based Retrieval with Deep Lake and OpenAI

Copyright 2024 Denis Rothman

**Updated to use Deep Lake v4 + LangChain integration.**

# 1. Installing the environment

*First run the following cells. Then run the notebook again cell by cell to explore the code.*

In [ ]:
!uv pip install "deeplake>=4.0" langchain-deeplake langchain-openai langchain langchain-community python-dotenv openai


In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["ACTIVELOOP_TOKEN"] = os.getenv("ACTIVELOOP_TOKEN")

print("API keys loaded.")


# Retrieval Augmented Generation

### Initiating the query process

**Set your org_id and dataset name below to match what you used in notebook 2.**

In [ ]:
# Set your Activeloop organization ID here
org_id = "mahmoudkamal01099s-organization"  # <-- Replace with your org ID if different
vector_store_path = f"al://{org_id}/first-workspace/my_table"
print(f"Loading from: {vector_store_path}")


In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_deeplake.vectorstores import DeeplakeVectorStore

# Initialize the same embeddings model used when creating the store
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# Load the existing vector store from the cloud
db = DeeplakeVectorStore(
    dataset_path=vector_store_path,
    embedding=embeddings,
)
print(f"Vector store loaded. Documents: {len(db.dataset)}")


## Input and Query Retrieval

### Retrieval query

In [ ]:
# User prompt
user_prompt = "Tell me about space exploration on the Moon and Mars."

# Perform similarity search using Deep Lake v4 / LangChain API
search_results = db.similarity_search_with_score(user_prompt, k=3)

print(f"Query: {user_prompt}\n")
for i, (doc, score) in enumerate(search_results):
    print(f"--- Result {i+1} | Score: {score:.4f} ---")
    print(f"Source: {doc.metadata.get('source', 'N/A')}")
    print(f"Text: {doc.page_content[:300]}...")
    print()


In [ ]:
print(user_prompt)


In [ ]:
# Extract the top result
top_doc, top_score = search_results[0]
top_text = top_doc.page_content.strip()
top_metadata = top_doc.metadata.get('source', 'N/A')

import textwrap
print("Top Search Result:")
print(f"Score: {top_score:.4f}")
print(f"Source: {top_metadata}")
print("Text:")
print(textwrap.fill(top_text, width=80))


## Augmented Input

In [ ]:
augmented_input = user_prompt + " " + top_text


In [ ]:
print(augmented_input)


# Generation and output

In [ ]:
from openai import OpenAI
import time

client = OpenAI()
gpt_model = "gpt-4o"
start_time = time.time()

def call_gpt4_with_full_text(itext):
    text_input = itext if isinstance(itext, str) else '\n'.join(itext)
    prompt = f"Please summarize or elaborate on the following content:\n{text_input}"

    try:
        response = client.chat.completions.create(
            model=gpt_model,
            messages=[
                {"role": "system", "content": "You are a space exploration expert."},
                {"role": "assistant", "content": "You can read the input and answer in detail."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.1
        )
        return response.choices[0].message.content
    except Exception as e:
        return str(e)

gpt4_response = call_gpt4_with_full_text(augmented_input)

response_time = time.time() - start_time
print(f"Response Time: {response_time:.2f} seconds")
print(gpt_model, "Response:", gpt4_response)


### Formatted response

In [ ]:
import textwrap
import re
from IPython.display import display, Markdown, HTML

def print_formatted_response(response):
    markdown_patterns = [
        r"^#+\s", r"^\*+", r"\*\*", r"_", r"\[.+\]\(.+\)", r"-\s", r"\`\`\`"
    ]
    if any(re.search(pattern, response, re.MULTILINE) for pattern in markdown_patterns):
        display(Markdown(response))
    else:
        wrapper = textwrap.TextWrapper(width=80)
        wrapped_text = wrapper.fill(text=response)
        print("Text Response:")
        print("--------------------")
        print(wrapped_text)
        print("--------------------\n")

print_formatted_response(gpt4_response)


# Evaluating the output with Cosine Similarity

with initial user prompt

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def calculate_cosine_similarity(text1, text2):
    vectorizer = TfidfVectorizer()
    tfidf = vectorizer.fit_transform([text1, text2])
    similarity = cosine_similarity(tfidf[0:1], tfidf[1:2])
    return similarity[0][0]

similarity_score = calculate_cosine_similarity(user_prompt, gpt4_response)
print(f"Cosine Similarity Score (user_prompt vs response): {similarity_score:.3f}")


with augmented user prompt

In [ ]:
similarity_score = calculate_cosine_similarity(augmented_input, gpt4_response)
print(f"Cosine Similarity Score (augmented_input vs response): {similarity_score:.3f}")


In [ ]:
!uv pip install sentence-transformers


In [ ]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('all-MiniLM-L6-v2')


In [ ]:
def calculate_cosine_similarity_with_embeddings(text1, text2):
    embeddings1 = model.encode(text1)
    embeddings2 = model.encode(text2)
    similarity = cosine_similarity([embeddings1], [embeddings2])
    return similarity[0][0]

similarity_score = calculate_cosine_similarity_with_embeddings(augmented_input, gpt4_response)
print(f"Cosine Similarity Score (sentence-transformers): {similarity_score:.3f}")
